# 05 - Naive RAG Experiments

Runs the baseline Naive RAG pipeline across different vector databases and embedding models.
This serves as the control experiment for comparison with Hybrid and Multi-Agent approaches.

**Pipeline:** Query → Dense Retrieval (Cosine/MMR) → LLM Generation → Answer  
**Variables:** Vector DB (Chroma, Qdrant, LanceDB), Embedding model (MiniLM, BGE)

In [ ]:
import sys
sys.path.append("..")

from langchain_huggingface import HuggingFaceEmbeddings
from src.data_loader import load_pubmedqa, build_mesh_lookup
from src.chunking import create_documents
from src.ingestion import ingest_to_chroma
from src.retrieval import get_cosine_retriever, get_mmr_retriever
from src.sampling import get_stratified_sample, create_golden_dataset
from src.llm_wrapper import get_groq_llm
from src.rag_pipeline import build_naive_rag_chain, run_rag
import config

## Setup

In [ ]:
df = load_pubmedqa()
mesh_lookup = build_mesh_lookup(df)

embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
llm = get_groq_llm()

print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {config.LLM_MODEL}")

## Prepare Evaluation Sample

In [ ]:
stratified_df = get_stratified_sample(df)
golden_dataset = create_golden_dataset(stratified_df)

stratified_df.to_csv(config.DATA_PROCESSED_DIR / "stratified_sample.csv", index=False)
golden_dataset.to_csv(config.DATA_PROCESSED_DIR / "golden_dataset_150.csv", index=False)
print(f"Golden dataset: {len(golden_dataset)} samples")

## Load Vector Store & Build RAG Chain

In [ ]:
documents, ids = create_documents(str(config.DATA_RAW_DIR), mesh_lookup)

vector_store = ingest_to_chroma(
    documents, ids, embeddings,
    db_name=f"{embedding_key}_pubmed_chroma"
)

cosine_retriever = get_cosine_retriever(vector_store)
rag_chain = build_naive_rag_chain(llm)

## Run Naive RAG (Cosine Retrieval)

In [ ]:
eval_dataset = run_rag(rag_chain, cosine_retriever, stratified_df, delay=20)
eval_dataset.to_csv(str(config.RESULTS_RAGAS_DIR / f"naive_rag_{embedding_key}_chroma_cosine.csv"))
print(f"Generated {len(eval_dataset)} answers")

## Run Naive RAG (MMR Retrieval)

In [ ]:
mmr_retriever = get_mmr_retriever(vector_store)
eval_dataset_mmr = run_rag(rag_chain, mmr_retriever, stratified_df, delay=20)
eval_dataset_mmr.to_csv(str(config.RESULTS_RAGAS_DIR / f"naive_rag_{embedding_key}_chroma_mmr.csv"))
print(f"Generated {len(eval_dataset_mmr)} answers (MMR)")

## Quick Test: Single Query

In [ ]:
test_question = stratified_df.iloc[0]["question"]
contexts = cosine_retriever.invoke(test_question)
result = rag_chain.invoke({"context": contexts, "question": test_question})

print(f"Q: {test_question}")
print(f"\nA: {result.content}")
print(f"\nRetrieved {len(contexts)} contexts")
for i, ctx in enumerate(contexts):
    print(f"  [{i+1}] {ctx.page_content[:100]}...")